# Module 1 · Lesson 06: Streaming Responses

**Streaming** delivers tokens one-by-one as they are generated, instead of waiting
for the full response. This is essential for building responsive AI applications.

## What you will learn
1. Basic streaming with OpenAI
2. Measuring **Time to First Token (TTFT)** and throughput
3. Streaming with Anthropic (different pattern!)
4. Streaming vs blocking — perceived latency
5. When to use streaming vs blocking

---
> ⚡ **Why streaming matters:** A 2-second delay feels slow. But seeing tokens
> appear instantly (even if the full response takes 2 seconds) feels fast and engaging.

In [6]:
import os

from pathlib import Path
from dotenv import  load_dotenv
from IPython.display import display, Markdown, clear_output

load_dotenv(Path.cwd().parent / "API_Verifications/.env")
    

from openai import OpenAI

client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

if client:
    print(f"✅ OpenAI client ready - using model {OPENAI_MODEL}")

✅ OpenAI client ready - using model gpt-4o-mini


---
## 1. Basic Streaming (OpenAI)

Set `stream=True` and iterate over the response. Each chunk contains a small piece of text.

In [4]:
# ── Example 1: Basic streaming ────────────────────────
print("🔄 Streaming response:\n")
 
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a short poem about Python programming."}],
    max_tokens=150,
    stream=True     # ← This enables streaming!
)
 
full_text = ""
for chunk in stream:
    # Each chunk may have content or be empty (keep-alive)
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        full_text += delta
 
print(f"\n\n📊 Total length: {len(full_text)} characters")

🔄 Streaming response:

In a world of code, where logic reigns,  
Python dances through digital lanes.  
With clean syntax and a gentle grace,  
It weaves solutions in cyberspace.  

Indentations form a structured embrace,  
While modules and libraries quicken the pace.  
From web apps to data, its versatility shines,  
A language that speaks in elegant lines.  

Loops and functions, a rhythm so sweet,  
In Python's embrace, our ideas take fleet.  
From novice to master, it welcomes us all,  
In the realm of programming, Python stands tall.

📊 Total length: 522 characters


> 💡 **Key Pattern:** `chunk.choices[0].delta.content` — note it's `delta` not `message`!
> The delta contains only the *new* content in each chunk.

---
## 2. Timing — TTFT and Throughput

Two critical metrics for streaming:
- **TTFT** (Time to First Token): How quickly the first token arrives
- **Throughput**: Tokens per second after streaming starts

In [ ]:
import time 

# ── Example 2: Timing metrics ─────────────────────────
prompt = "Explain what machine learning is in 3 sentences."

start = time.perf_counter()
ttft = None
token_count = 0
full_text = ""

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    stream=True
)

print("🔄 Streaming with timing:\n")

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000  # ms
        print(delta, end="", flush=True)
        full_text += delta
        token_count += 1

total_ms = (time.perf_counter() - start) * 1000
tps = token_count / (total_ms / 1000) if total_ms > 0 else 0

print(f"\n\n{'─'*40}")
print(f"⏱  Time to First Token (TTFT): {ttft:.0f} ms")
print(f"⏱  Total time:                 {total_ms:.0f} ms")
print(f"📊 Tokens received:             {token_count}")
print(f"⚡ Throughput:                  {tps:.1f} tokens/sec")

🔄 Streaming with timing:

Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on tasks without being explicitly programmed. It involves the use of algorithms and statistical models to identify patterns and make predictions based on input data. Through training on large datasets, machine learning systems can adapt and generalize their insights to new, unseen situations.

────────────────────────────────────────
⏱  Time to First Token (TTFT): 558 ms
⏱  Total time:                 1393 ms
📊 Tokens received:             68
⚡ Throughput:                  48.8 tokens/sec


---
## 3. Streaming with Anthropic

Anthropic uses a **context manager** pattern instead of `stream=True`:

In [5]:
# ── Example 3: Anthropic streaming ────────────────────
try:
    import anthropic
    ac = anthropic.Anthropic()

    print("🔄 Claude streaming:\n")

    with ac.messages.stream(
        model="claude-sonnet-4-6",
        max_tokens=150,
        messages=[{"role": "user", "content": "Write a haiku about AI."}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)

    print("\n\n✅ Anthropic streaming works!")

except Exception as e:
    print(f"⚠️ Anthropic not available: {e}")
    print("   That's OK — OpenAI streaming is the primary focus.")

🔄 Claude streaming:

Here's a haiku about AI:

*Silent mind of code*
*Learning from a million words*
*Dreams it cannot feel*

✅ Anthropic streaming works!


### API Comparison: Streaming

```python
# OpenAI streaming
stream = client.chat.completions.create(
    ..., stream=True
)
for chunk in stream:
    text = chunk.choices[0].delta.content

# Anthropic streaming
with client.messages.stream(...) as stream:
    for text in stream.text_stream:
        print(text)
```

---
## 4. Streaming vs Blocking — Latency Comparison

In [9]:
import time
from time import perf_counter
# ── Example 4: Streaming vs Blocking ──────────────────
prompt = "List 5 benefits of learning Python."

# Method 1: Blocking (wait for full response)
start = time.perf_counter()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)
blocking_time = (time.perf_counter() - start) * 1000

# Method 2: Streaming (measure TTFT)
start = time.perf_counter()
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    stream=True
)
streaming_ttft = None
for chunk in stream:
    if chunk.choices[0].delta.content and streaming_ttft is None:
        streaming_ttft = (time.perf_counter() - start) * 1000
streaming_total = (time.perf_counter() - start) * 1000

# Results
print("⏱  Perceived Latency Comparison")
print(f"{'─'*45}")
print(f"Blocking:  User waits {blocking_time:.0f} ms to see ANYTHING")
print(f"Streaming: User sees first token in {streaming_ttft:.0f} ms")
print(f"           (Full response in {streaming_total:.0f} ms)")
print(f"\n💡 Perceived improvement: {blocking_time - streaming_ttft:.0f} ms faster!")

⏱  Perceived Latency Comparison
─────────────────────────────────────────────
Blocking:  User waits 3643 ms to see ANYTHING
Streaming: User sees first token in 1062 ms
           (Full response in 3566 ms)

💡 Perceived improvement: 2582 ms faster!


---
## 5. Real-World Exercise: Scrape & Stream

Let's combine streaming with a practical use case: **scrape a webpage and stream
a summary of its content**. This is inspired by community projects that scrape
news/JIRA/email and summarize with LLMs.

```
Web Page --> requests + BeautifulSoup --> Raw Text --> GPT (stream) --> Summary
```

In [3]:
import requests
from bs4 import BeautifulSoup

def scape_text(url: str, max_chars: int = 3000) -> str:
    """ Fetch a URL and extract clean content"""
    headers = {"User-Agent": "Mozilla/5.0 (Educational Bot)"}
    r = requests.get(url, headers= headers, timeout= 10)
    soup = BeautifulSoup(r.text, "html.parser")

    # Remove noise
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator= "\n", strip= True)
    return text[:max_chars]

try:
    page_text = scape_text("https://diaviou.aueb.gr/programs/36148-AI-for-Developers--Design,-Build,-Deploy-LLM-powered-Applications-1001200")
    print(f"Scraped: {len(page_text)} characters")
    print(f"Preview: {page_text[:200]}...")
except ValueError as e:
    # fallback if no internet connection...
    page_text = """Python is a programming language that lets you work quickly
and integrate systems more effectively. Python is powerful and fast, plays well
with others, runs everywhere, is friendly and easy to learn, and is open.
Python can be used for web development, data analysis, machine learning,
scientific computing, automation, and more."""
   
    print(f"Using fallback text ({len(page_text)} chars): {e}")

Scraped: 3000 characters
Preview: AI for Developers: Design, Build, Deploy LLM-powered Applications
210 8203913
secretariat@diaviou.aueb.gr
ΠΡΟΓΡΑΜΜΑΤΑ
Αρχική
ΠΡΟΓΡΑΜΜΑΤΑ
ΠΡΟΓΡΑΜΜΑΤΑ
Εκτύπωση
ΠΡΟΓΡΑΜΜΑΤΑ
AI for Developers: Design, Bui...


In [9]:
import time
print("Streaming summury of scraped content:\n")
print("=" * 50)
start = time.perf_counter()
ttft = None
full_summury = ""
md_display = display(Markdown(""), display_id= True)

stream = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages= [
        {"role": "system", "content": "Summarize web page content in 3-5 bullet points."
                                        "Be concise and highlight key facts."},
        {"role": "user", "content": f"Summarize this web page content: \n\n {page_text}"}
    ],
    max_tokens= 500,
    stream= True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000
        md_display.update(Markdown(full_summury))
        full_summury += delta
total_ms = (time.perf_counter() - start) * 1000

print(f"\n\n {"=" * 50}")
print(f"TTFT: {ttft:.0f} ms | Total: {total_ms:.0f} ms")
print(f"Input: {len(page_text)} chars scraped | Output: {len(full_summury)} chars summary")

Streaming summury of scraped content:



- The program "AI for Developers" provides a comprehensive introduction to Artificial Intelligence, focusing on Large Language Models (LLMs).
- Participants will learn to integrate AI models into applications using Python and FastAPI, design effective prompts, and develop solutions leveraging Retrieval-Augmented Generation (RAG) and autonomous AI agents.
- Two attendance options are available: in-person or live streaming, with the latter allowing for greater flexibility and access to course materials.
- Upon completion, participants will have practical skills in AI application development, prompt engineering, and working with open-source models, concluding with a project that showcases their capabilities.
- The course includes a review of Python and FastAPI basics, totaling 18 hours of instruction



TTFT: 647 ms | Total: 3721 ms
Input: 3000 chars scraped | Output: 809 chars summary


> **Exercise:** Modify the scraper to fetch a news article of your choice.
> Compare the streaming summary with a blocking call — which feels faster to the user?
> Try adding `temperature=0` for factual summaries vs `temperature=0.7` for creative ones.

---
## When to Use Each Pattern

| Pattern | Use When |
|---------|----------|
| **Streaming** | Chat interfaces, long responses, user-facing apps |
| **Blocking** | Background processing, batch jobs, structured output (JSON) |

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **TTFT** | Time to first token — the key UX metric |
| **Throughput** | Tokens/second — depends on model and load |
| **OpenAI** | `stream=True` + iterate chunks |
| **Anthropic** | `messages.stream()` context manager |
| **delta vs message** | Streaming gives `.delta.content`, blocking gives `.message.content` |
| **Scrape + Stream** | Combine web scraping with streaming for real-time summaries |

---
**Next:** `07_response_inspector.ipynb` — Understand the full response metadata